# Assignment 3 — Deep Learning (NNDL)

> Spec PDF: `/Users/tahamajs/Documents/uni/LLM/Deep_UT/This_year/CA3/description/NNDL_Assignment3.pdf`

This notebook contains complete, runnable code for medical image segmentation using U-Net architecture on IBSR brain segmentation dataset.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import math
import random
import time
from pathlib import Path
from glob import glob

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

import sys
sys.path.append('../dataset')
from Q1_dataprep import IBSRPatchDataset, get_slice_data, pad_slice

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)

device = torch.device('cpu')
if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')

print(f"Using device: {device}")

# Question 1 — Medical Image Segmentation with U-Net

## 1-1. Dataset Preparation and Loading

In [ ]:
# Configuration
CONFIG = {
    'data_dir': '../dataset',  # Update this path to your dataset location
    'batch_size': 16,
    'epochs': 50,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'num_workers': 2,
    'seed': 42,
    'save_dir': './checkpoints',
    'num_classes': 2,  # Background and foreground (brain tissue)
    'patch_size': 128,
    'in_channels': 1
}

Path(CONFIG['save_dir']).mkdir(parents=True, exist_ok=True)

# Update these paths to point to your IBSR dataset
# Example structure:
# dataset/
#   volumes/
#     volume1.nii.gz
#     volume2.nii.gz
#   segmentations/
#     segmentation1.nii.gz
#     segmentation2.nii.gz

# You'll need to provide the actual paths to your volume and segmentation files
# volume_files = sorted(glob(f"{CONFIG['data_dir']}/volumes/*.nii.gz"))
# segmentation_files = sorted(glob(f"{CONFIG['data_dir']}/segmentations/*.nii.gz"))

# For now, we'll create a placeholder - replace with actual paths
volume_files = []  # Add your volume file paths here
segmentation_files = []  # Add your segmentation file paths here

print(f"Found {len(volume_files)} volume files")
print(f"Found {len(segmentation_files)} segmentation files")

In [ ]:
# Split dataset into train/val/test
if len(volume_files) > 0:
    # Assuming we have multiple subjects
    indices = list(range(len(volume_files)))
    random.shuffle(indices)
    
    train_size = int(0.7 * len(indices))
    val_size = int(0.15 * len(indices))
    
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]
    
    train_volumes = [volume_files[i] for i in train_indices]
    train_segs = [segmentation_files[i] for i in train_indices]
    
    val_volumes = [volume_files[i] for i in val_indices]
    val_segs = [segmentation_files[i] for i in val_indices]
    
    test_volumes = [volume_files[i] for i in test_indices]
    test_segs = [segmentation_files[i] for i in test_indices]
    
    # Create datasets
    train_dataset = IBSRPatchDataset(train_volumes, train_segs)
    val_dataset = IBSRPatchDataset(val_volumes, val_segs)
    test_dataset = IBSRPatchDataset(test_volumes, test_segs)
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True, 
        num_workers=CONFIG['num_workers'],
        pin_memory=True if device.type == 'cuda' else False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False, 
        num_workers=CONFIG['num_workers'],
        pin_memory=True if device.type == 'cuda' else False
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False, 
        num_workers=CONFIG['num_workers'],
        pin_memory=True if device.type == 'cuda' else False
    )
    
    print(f"Train samples: {len(train_dataset)}")
    print(f"Val samples: {len(val_dataset)}")
    print(f"Test samples: {len(test_dataset)}")
else:
    print("Please update volume_files and segmentation_files with actual paths to your dataset")

## 1-2. U-Net Model Architecture

In [ ]:
class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # input is CHW
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=False):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits


# Create model
model = UNet(n_channels=CONFIG['in_channels'], n_classes=CONFIG['num_classes'], bilinear=False)
model = model.to(device)

# Print model summary
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model parameters: {count_parameters(model):,}")
print(f"\nModel architecture:")
print(model)

## 1-3. Loss Functions and Metrics

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        # Apply softmax to get probabilities
        inputs = F.softmax(inputs, dim=1)
        
        # One-hot encode targets
        num_classes = inputs.shape[1]
        targets_one_hot = F.one_hot(targets, num_classes).permute(0, 3, 1, 2).float()
        
        # Flatten tensors
        inputs_flat = inputs.view(inputs.size(0), inputs.size(1), -1)
        targets_flat = targets_one_hot.view(targets_one_hot.size(0), targets_one_hot.size(1), -1)
        
        # Calculate Dice coefficient for each class
        intersection = (inputs_flat * targets_flat).sum(dim=2)
        union = inputs_flat.sum(dim=2) + targets_flat.sum(dim=2)
        
        dice = (2. * intersection + self.smooth) / (union + self.smooth)
        dice_loss = 1 - dice.mean()
        
        return dice_loss


class CombinedLoss(nn.Module):
    """Combination of Dice Loss and Cross Entropy Loss"""
    def __init__(self, dice_weight=0.5, ce_weight=0.5, smooth=1.0):
        super(CombinedLoss, self).__init__()
        self.dice_weight = dice_weight
        self.ce_weight = ce_weight
        self.dice_loss = DiceLoss(smooth=smooth)
        self.ce_loss = nn.CrossEntropyLoss()

    def forward(self, inputs, targets):
        dice = self.dice_loss(inputs, targets)
        ce = self.ce_loss(inputs, targets)
        return self.dice_weight * dice + self.ce_weight * ce


# Loss function
criterion = CombinedLoss(dice_weight=0.5, ce_weight=0.5)

# Metrics functions
def calculate_dice_score(pred, target, num_classes, smooth=1.0):
    """Calculate Dice score for each class"""
    pred_one_hot = F.one_hot(pred, num_classes).permute(0, 3, 1, 2).float()
    target_one_hot = F.one_hot(target, num_classes).permute(0, 3, 1, 2).float()
    
    pred_flat = pred_one_hot.view(pred_one_hot.size(0), pred_one_hot.size(1), -1)
    target_flat = target_one_hot.view(target_one_hot.size(0), target_one_hot.size(1), -1)
    
    intersection = (pred_flat * target_flat).sum(dim=2)
    union = pred_flat.sum(dim=2) + target_flat.sum(dim=2)
    
    dice = (2. * intersection + smooth) / (union + smooth)
    return dice.mean(dim=0)  # Average over batch, return per-class scores


def calculate_iou(pred, target, num_classes, smooth=1.0):
    """Calculate IoU (Intersection over Union) for each class"""
    pred_one_hot = F.one_hot(pred, num_classes).permute(0, 3, 1, 2).float()
    target_one_hot = F.one_hot(target, num_classes).permute(0, 3, 1, 2).float()
    
    pred_flat = pred_one_hot.view(pred_one_hot.size(0), pred_one_hot.size(1), -1)
    target_flat = target_one_hot.view(target_one_hot.size(0), target_one_hot.size(1), -1)
    
    intersection = (pred_flat * target_flat).sum(dim=2)
    union = pred_flat.sum(dim=2) + target_flat.sum(dim=2) - intersection
    
    iou = (intersection + smooth) / (union + smooth)
    return iou.mean(dim=0)  # Average over batch, return per-class scores


def calculate_pixel_accuracy(pred, target):
    """Calculate pixel-wise accuracy"""
    correct = (pred == target).sum().item()
    total = target.numel()
    return correct / total

## 1-4. Training Loop

In [ ]:
# Optimizer
optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'train_dice': [],
    'val_dice': [],
    'train_iou': [],
    'val_iou': [],
    'train_acc': [],
    'val_acc': []
}

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    running_acc = 0.0
    num_batches = 0
    
    for batch_idx, (images, masks) in enumerate(loader):
        images = images.to(device)
        masks = masks.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Calculate metrics
        preds = torch.argmax(outputs, dim=1)
        dice_scores = calculate_dice_score(preds, masks, CONFIG['num_classes'])
        iou_scores = calculate_iou(preds, masks, CONFIG['num_classes'])
        acc = calculate_pixel_accuracy(preds, masks)
        
        running_loss += loss.item()
        running_dice += dice_scores.mean().item()
        running_iou += iou_scores.mean().item()
        running_acc += acc
        num_batches += 1
        
        if (batch_idx + 1) % 10 == 0:
            print(f'  Batch {batch_idx + 1}/{len(loader)}, Loss: {loss.item():.4f}, '
                  f'Dice: {dice_scores.mean().item():.4f}, IoU: {iou_scores.mean().item():.4f}, '
                  f'Acc: {acc:.4f}')
    
    return (running_loss / num_batches, running_dice / num_batches, 
            running_iou / num_batches, running_acc / num_batches)


def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    running_acc = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            preds = torch.argmax(outputs, dim=1)
            dice_scores = calculate_dice_score(preds, masks, CONFIG['num_classes'])
            iou_scores = calculate_iou(preds, masks, CONFIG['num_classes'])
            acc = calculate_pixel_accuracy(preds, masks)
            
            running_loss += loss.item()
            running_dice += dice_scores.mean().item()
            running_iou += iou_scores.mean().item()
            running_acc += acc
            num_batches += 1
    
    return (running_loss / num_batches, running_dice / num_batches, 
            running_iou / num_batches, running_acc / num_batches)

In [ ]:
# Training loop
if len(volume_files) > 0:
    best_val_dice = 0.0
    patience_counter = 0
    patience = 10
    
    print("Starting training...")
    print(f"Total epochs: {CONFIG['epochs']}")
    print("-" * 60)
    
    for epoch in range(CONFIG['epochs']):
        print(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")
        print(f"Learning rate: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Train
        train_loss, train_dice, train_iou, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device
        )
        
        # Validate
        val_loss, val_dice, val_iou, val_acc = validate_epoch(
            model, val_loader, criterion, device
        )
        
        # Update learning rate
        scheduler.step(val_loss)
        
        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_dice'].append(train_dice)
        history['val_dice'].append(val_dice)
        history['train_iou'].append(train_iou)
        history['val_iou'].append(val_iou)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"\nTrain - Loss: {train_loss:.4f}, Dice: {train_dice:.4f}, "
              f"IoU: {train_iou:.4f}, Acc: {train_acc:.4f}")
        print(f"Val   - Loss: {val_loss:.4f}, Dice: {val_dice:.4f}, "
              f"IoU: {val_iou:.4f}, Acc: {val_acc:.4f}")
        
        # Save best model
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_dice': val_dice,
                'history': history
            }, f"{CONFIG['save_dir']}/best_model.pth")
            print(f"✓ Saved best model (Dice: {best_val_dice:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\nEarly stopping at epoch {epoch + 1}")
                break
    
    print("\n" + "=" * 60)
    print("Training completed!")
    print(f"Best validation Dice score: {best_val_dice:.4f}")
else:
    print("Skipping training - please provide dataset paths")

## 1-5. Training Curves Visualization

In [ ]:
if len(history['train_loss']) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss curves
    axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
    axes[0, 0].plot(history['val_loss'], label='Val Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Dice score curves
    axes[0, 1].plot(history['train_dice'], label='Train Dice', linewidth=2)
    axes[0, 1].plot(history['val_dice'], label='Val Dice', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Dice Score')
    axes[0, 1].set_title('Training and Validation Dice Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # IoU curves
    axes[1, 0].plot(history['train_iou'], label='Train IoU', linewidth=2)
    axes[1, 0].plot(history['val_iou'], label='Val IoU', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('IoU Score')
    axes[1, 0].set_title('Training and Validation IoU Score')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Accuracy curves
    axes[1, 1].plot(history['train_acc'], label='Train Acc', linewidth=2)
    axes[1, 1].plot(history['val_acc'], label='Val Acc', linewidth=2)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Pixel Accuracy')
    axes[1, 1].set_title('Training and Validation Pixel Accuracy')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/training_curves.png", dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No training history available")

## 1-6. Test Set Evaluation

In [ ]:
if len(volume_files) > 0:
    # Load best model
    checkpoint = torch.load(f"{CONFIG['save_dir']}/best_model.pth")
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded best model from epoch {checkpoint['epoch'] + 1}")
    
    # Evaluate on test set
    model.eval()
    test_loss = 0.0
    test_dice_scores = []
    test_iou_scores = []
    test_acc_scores = []
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            preds = torch.argmax(outputs, dim=1)
            
            # Calculate per-class metrics
            dice_scores = calculate_dice_score(preds, masks, CONFIG['num_classes'])
            iou_scores = calculate_iou(preds, masks, CONFIG['num_classes'])
            acc = calculate_pixel_accuracy(preds, masks)
            
            test_loss += loss.item()
            test_dice_scores.append(dice_scores.cpu().numpy())
            test_iou_scores.append(iou_scores.cpu().numpy())
            test_acc_scores.append(acc)
            
            all_preds.append(preds.cpu())
            all_targets.append(masks.cpu())
    
    # Average metrics
    test_loss /= len(test_loader)
    avg_dice = np.mean([d.mean() for d in test_dice_scores])
    avg_iou = np.mean([i.mean() for i in test_iou_scores])
    avg_acc = np.mean(test_acc_scores)
    
    # Per-class metrics
    per_class_dice = np.mean(test_dice_scores, axis=0)
    per_class_iou = np.mean(test_iou_scores, axis=0)
    
    print("\n" + "=" * 60)
    print("TEST SET EVALUATION")
    print("=" * 60)
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Average Dice Score: {avg_dice:.4f}")
    print(f"Average IoU Score: {avg_iou:.4f}")
    print(f"Pixel Accuracy: {avg_acc:.4f}")
    print("\nPer-class metrics:")
    for i in range(CONFIG['num_classes']):
        print(f"  Class {i}: Dice={per_class_dice[i]:.4f}, IoU={per_class_iou[i]:.4f}")
    print("=" * 60)
else:
    print("Skipping evaluation - please provide dataset paths")

## 1-7. Visualization of Predictions

In [ ]:
if len(volume_files) > 0:
    # Visualize some test predictions
    model.eval()
    num_samples = min(6, len(test_loader))
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    with torch.no_grad():
        for idx, (images, masks) in enumerate(test_loader):
            if idx >= num_samples:
                break
            
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            
            # Take first image from batch
            img = images[0, 0].cpu().numpy()
            mask = masks[0].cpu().numpy()
            pred = preds[0].cpu().numpy()
            
            # Plot
            axes[idx, 0].imshow(img, cmap='gray')
            axes[idx, 0].set_title('Input Image')
            axes[idx, 0].axis('off')
            
            axes[idx, 1].imshow(mask, cmap='jet')
            axes[idx, 1].set_title('Ground Truth')
            axes[idx, 1].axis('off')
            
            axes[idx, 2].imshow(pred, cmap='jet')
            axes[idx, 2].set_title('Prediction')
            axes[idx, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/predictions.png", dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Skipping visualization - please provide dataset paths")